In [4]:
import pandas as pd

In [8]:
data = pd.read_csv("datasets/CapnoBase/data/csv/0333_8min_labels.csv")
print(data["ecg_peak_x"])
print(data["ecg_artif_x"])

0     185 459 732 1004 1278 1550 1824 2100 2377 265...
Name: ecg_peak_x, dtype: str
0   NaN
Name: ecg_artif_x, dtype: float64


In [22]:
def get_raw_data(data_path):
    """ Load data from the specified path. """
    files = ["Breaths", "Numerics", "Signals"]
    num_of_subjects = 53
    
    data_list = []
    
    for subject_id in range(1, num_of_subjects + 1):
        subject_data = {"id": f"bidmc_{subject_id:02d}"}
        for file in files:
            file_path = f"{data_path}/bidmc_{subject_id:02d}_{file}.csv"
            df = pd.read_csv(file_path)
            
            # Strip whitespace from column names
            df.columns = df.columns.str.strip()
            
            subject_data[file] = df
        data_list.append(subject_data)
        
    return data_list

data_path = "datasets/bidmc-ppg-and-respiration-dataset-1.0.0/bidmc_csv"
data = get_raw_data(data_path)

In [23]:
data[0]["Signals"].head()

,Time [s],RESP,PLETH,V,AVR,II
0,0.000,0.35386,0.43597,0.52549,0.30392,0.72549
1,0.008,0.35679,0.43206,0.51961,0.33529,0.67059
2,0.016,0.35875,0.42815,0.51569,0.37451,0.60980
3,0.024,0.36168,0.42424,0.50588,0.41961,0.55098
4,0.032,0.36364,0.42131,0.50980,0.44902,0.50000


In [24]:
data[0]["Breaths"].head()

,breaths ann1 [signal sample no],breaths ann2 [signal sample no]
0,189,187
1,476,472
2,737,747
3,1037,1042
4,1324,1322


In [26]:
data[0]["Numerics"].head()

,Time [s],HR,PULSE,RESP,SpO2
0,0,94,93.0,25,97.0
1,1,94,93.0,25,97.0
2,2,94,93.0,25,97.0
3,3,92,93.0,26,97.0
4,4,93,93.0,26,97.0


In [27]:
print(data[0]["Numerics"]["HR"])

0      94
1      94
2      94
3      92
4      93
       ..
476    90
477    90
478    90
479    90
480    90
Name: HR, Length: 481, dtype: int64


In [31]:
def process_data(data):
    processed_data = []

    for subject_data in data:
        subject_id = subject_data["id"]
        signals_df = subject_data["Signals"]
        numerics_df = subject_data["Numerics"]
        
        # Assuming the PPG signal is in the 'PPG' column and HR in 'HR' column
        ppg_signal = signals_df['PLETH'].values
        hr_values = numerics_df['HR'].values
        
        # Segment the data into 1-second intervals (assuming 125 samples per second)
        num_segments = len(ppg_signal) // 125
        
        for i in range(num_segments):
            start_idx = i * 125
            end_idx = start_idx + 125
            
            segment_ppg = ppg_signal[start_idx:end_idx].tolist()
            segment_hr = hr_values[i] if i < len(hr_values) else None
            
            segment_data = {
                "id": subject_id,
                "start": i,
                "end": i + 1,
                "PPG": segment_ppg,
                "HR": segment_hr
            }
            
            processed_data.append(segment_data)
    
    return processed_data

In [32]:
processed_data = process_data(data)

print(processed_data[0])

{'id': 'bidmc_01', 'start': 0, 'end': 1, 'PPG': [0.43597, 0.43206, 0.42815, 0.42424, 0.42131, 0.41838, 0.4174, 0.41642, 0.41544, 0.41447, 0.41251, 0.41153, 0.40958, 0.40762, 0.40665, 0.40665, 0.40665, 0.40762, 0.4086, 0.4086, 0.41056, 0.41153, 0.41153, 0.41056, 0.4086, 0.40665, 0.40469, 0.40469, 0.40567, 0.4086, 0.41251, 0.42033, 0.43206, 0.45259, 0.4741, 0.4956, 0.51808, 0.5347, 0.55034, 0.56403, 0.57576, 0.58456, 0.59238, 0.59629, 0.59922, 0.60117, 0.60215, 0.60313, 0.60313, 0.60117, 0.59824, 0.59335, 0.58553, 0.57771, 0.57185, 0.56207, 0.54741, 0.53372, 0.52102, 0.50929, 0.50244, 0.49756, 0.4956, 0.49462, 0.49365, 0.49365, 0.49365, 0.49169, 0.48876, 0.4868, 0.48387, 0.48094, 0.47801, 0.47507, 0.47116, 0.46823, 0.46334, 0.45846, 0.45357, 0.44868, 0.44379, 0.43891, 0.435, 0.43109, 0.4262, 0.42326, 0.42033, 0.41838, 0.41642, 0.41447, 0.41447, 0.41349, 0.41251, 0.41251, 0.41251, 0.41251, 0.41251, 0.41349, 0.41544, 0.4174, 0.41935, 0.42033, 0.42131, 0.42131, 0.42033, 0.41935, 0.41642, 0.

In [34]:
from pathlib import Path

def get_raw_data_from_csv(data_path):
    files = ["labels", "meta", "param", "reference", "signal", "SFresults"]

    # Automatically get all subject IDs
    subject_ids = sorted({
        f.name.split("_8min_")[0]
        for f in data_path.glob("*_8min_*.csv")
    })

    # print(f"Found {len(subject_ids)} subjects.")

    data_list = []

    for subject_id in subject_ids:
        subject_data = {"id": f"capnobase_{subject_id}"}

        for file in files:
            file_path = data_path / f"{subject_id}_8min_{file}.csv"

            if not file_path.exists():
                # print(f"Warning: {file_path.name} not found.")
                continue

            df = pd.read_csv(file_path)
            df.columns = df.columns.str.strip()

            subject_data[file] = df

        data_list.append(subject_data)

    return data_list


data_path = Path("datasets/CapnoBase/data/csv")
data = get_raw_data_from_csv(data_path)

In [39]:
print(f"Loaded data for {len(data)} subjects.")
print(data[0].keys())

print([data[i]["id"] for i in range(len(data))])

Loaded data for 42 subjects.
dict_keys(['id', 'labels', 'meta', 'param', 'reference', 'signal', 'SFresults'])
['capnobase_0009', 'capnobase_0015', 'capnobase_0016', 'capnobase_0018', 'capnobase_0023', 'capnobase_0028', 'capnobase_0029', 'capnobase_0030', 'capnobase_0031', 'capnobase_0032', 'capnobase_0035', 'capnobase_0038', 'capnobase_0103', 'capnobase_0104', 'capnobase_0105', 'capnobase_0115', 'capnobase_0121', 'capnobase_0122', 'capnobase_0123', 'capnobase_0125', 'capnobase_0127', 'capnobase_0128', 'capnobase_0133', 'capnobase_0134', 'capnobase_0142', 'capnobase_0147', 'capnobase_0148', 'capnobase_0149', 'capnobase_0150', 'capnobase_0309', 'capnobase_0311', 'capnobase_0312', 'capnobase_0313', 'capnobase_0322', 'capnobase_0325', 'capnobase_0328', 'capnobase_0329', 'capnobase_0330', 'capnobase_0331', 'capnobase_0332', 'capnobase_0333', 'capnobase_0370']


In [40]:
print(data[0]["signal"].head())

   co2_y  pleth_y  ecg_y
0   1.50    -0.64  -0.31
1   1.54    -0.64  -0.34
2   1.59    -0.67  -0.31
3   1.64    -0.69  -0.26
4   1.68    -0.72  -0.18


In [41]:
print(data[0]["labels"].head())

   co2_artif_x                                     co2_startexp_x  \
0          NaN   974 1990 2974 3966 4990 5953 6998 7988 8995 9...   

                                     co2_startinsp_x  pleth_artif_x  \
0   659 1660 2661 3647 4658 5636 6670 7655 8676 9...            NaN   

                                        pleth_peak_x  \
0   59 242 422 611 800 983 1157 1334 1517 1703 18...   

                                          ecg_peak_x  ecg_artif_x  units_x  
0   164 346 532 718 898 1078 1258 1440 1621 1800 ...          NaN  samples  


In [43]:
print(data[0]["param"].head())

   samplingrate_pleth  samplingrate_co2  samplingrate_ecg    case_id  \
0                 300               300               300  0009_8min   

  case_ventilation  
0      spontaneous  


In [44]:
print(data[0]["labels"]["ecg_peak_x"])
print(type(data[0]["labels"]["ecg_peak_x"]))

0     164 346 532 718 898 1078 1258 1440 1621 1800 ...
Name: ecg_peak_x, dtype: str
<class 'pandas.Series'>
